# Spaceship Titanic

[Kaggle Competition](www.kaggle.com/competitions/spaceship-titanic).

Welcome to the year 2912, where your data science skills are needed to solve a cosmic mystery. We've received a transmission from four lightyears away and things aren't looking good.

The Spaceship Titanic was an interstellar passenger liner launched a month ago. With almost 13,000 passengers on board, the vessel set out on its maiden voyage transporting emigrants from our solar system to three newly habitable exoplanets orbiting nearby stars.

While rounding Alpha Centauri en route to its first destination—the torrid 55 Cancri E—the unwary Spaceship Titanic collided with a spacetime anomaly hidden within a dust cloud. Sadly, it met a similar fate as its namesake from 1000 years before. Though the ship stayed intact, almost half of the passengers were transported to an alternate dimension!

## Data importation

In [75]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.model_selection import cross_validate
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import lightgbm as lgb

In [2]:
train_df = pd.read_csv('data/titanic_train.csv')
test_df = pd.read_csv('data/titanic_test.csv')

train_df

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [3]:
X_train = train_df.drop(columns=['Transported'])
y_train = train_df.Transported

## Exploration

In [4]:
X_train.drop_duplicates(inplace=True)
X_train.shape

(8693, 13)

In [5]:
X_train.select_dtypes(include='object').isnull().sum().sort_values(ascending=False) / len(X_train)

CryoSleep      0.024963
VIP            0.023352
HomePlanet     0.023122
Name           0.023007
Cabin          0.022892
Destination    0.020936
PassengerId    0.000000
dtype: float64

In [6]:
feat_ordinal_dict = {
    'CryoSleep': ['missing', 'False', 'True'],
    'VIP': ['missing', 'False', 'True'],
    'HomePlanet': ['missing', 'Europa', 'Earth', 'Mars'],
    'Destination': ['missing', 'TRAPPIST-1e', 'PSO J318.5-22', '55 Cancri e']
}

In [7]:
ordinal_features = list(feat_ordinal_dict.keys())
ordinal_values = list(feat_ordinal_dict.values())

encode_ordinal = OrdinalEncoder(
    categories=ordinal_values,
    dtype=np.int64,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

prepoc_ordinal = Pipeline([
    ('nan_imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ordinal_encoded', encode_ordinal),
    ('scaler', RobustScaler())
])

prepoc_ordinal

Pipeline(steps=[('nan_imputer',
                 SimpleImputer(fill_value='missing', strategy='constant')),
                ('ordinal_encoded',
                 OrdinalEncoder(categories=[['missing', 'False', 'True'],
                                            ['missing', 'False', 'True'],
                                            ['missing', 'Europa', 'Earth',
                                             'Mars'],
                                            ['missing', 'TRAPPIST-1e',
                                             'PSO J318.5-22', '55 Cancri e']],
                                dtype=<class 'numpy.int64'>,
                                handle_unknown='use_encoded_value',
                                unknown_value=-1)),
                ('scaler', RobustScaler())])

## Numerical features

In [8]:
numerical_features = sorted(X_train.select_dtypes(include=np.number).columns)

preproc_numerical = Pipeline([
    ('nan_imputer', SimpleImputer(strategy='constant', fill_value=-1)),
    ('scaler', MinMaxScaler())
])

preproc_numerical

Pipeline(steps=[('nan_imputer',
                 SimpleImputer(fill_value=-1, strategy='constant')),
                ('scaler', MinMaxScaler())])

## Categorical features

In [9]:
categorical_features = sorted(X_train.select_dtypes(include='object').columns)

preproc_categorical = Pipeline([
    ('nan_imputer', SimpleImputer(strategy="constant", fill_value='None')),
    ('encode', OneHotEncoder(sparse_output=True, handle_unknown='ignore', drop='if_binary'))
])

preproc_categorical

Pipeline(steps=[('nan_imputer',
                 SimpleImputer(fill_value='None', strategy='constant')),
                ('encode',
                 OneHotEncoder(drop='if_binary', handle_unknown='ignore'))])

In [10]:
new_categorical_features = []
for feature in categorical_features:
    if feature not in ordinal_features: new_categorical_features.append(feature)

categorical_features = new_categorical_features
categorical_features

['Cabin', 'Name', 'PassengerId']

## Preprocessing Pipe

In [11]:
preproc_pipe = ColumnTransformer([
    ('ordinal_preprocessor', prepoc_ordinal, ordinal_features),
    ('numerical_preprocessor', preproc_numerical, numerical_features),
    ('categorical_preprocessor', preproc_categorical, categorical_features)
])

In [12]:
preproc_pipe.fit_transform(X_train)

<8693x23738 sparse matrix of type '<class 'numpy.float64'>'
	with 84404 stored elements in Compressed Sparse Row format>

## Logistic Regression

In [25]:
log_pipe = Pipeline([
    ('preprocessing', preproc_pipe),
    ('modeling', LogisticRegression(random_state=42))
])

log_pipe

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('ordinal_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('ordinal_encoded',
                                                                   OrdinalEncoder(categories=[['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'Europa',
                                                                                               'Earth',
                                                                                               'Mars'],
                                                                                              ['missing',
                                                                                               'TRAPPIST-1e',
                                                                                               'PSO '
                                                                                               'J318.5...
                                                                   MinMaxScaler())]),
                                                  ['Age', 'FoodCourt',
                                                   'RoomService',
                                                   'ShoppingMall', 'Spa',
                                                   'VRDeck']),
                                                 ('categorical_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='None',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(drop='if_binary',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Cabin', 'Name',
                                                   'PassengerId'])])),
                ('modeling', LogisticRegression(random_state=42))])

In [26]:
log_baseline_cv = cross_validate(log_pipe,
                                 X_train, y_train,
                                 cv=5,
                                 scoring='accuracy',
                                 n_jobs=-1
                                 )
log_baseline_cv

/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as

{'fit_time': array([0.2222507 , 0.25711274, 0.24213624, 0.2543776 , 0.24345422]),
 'score_time': array([0.1206696 , 0.10735106, 0.09282875, 0.12491703, 0.10729504]),
 'test_score': array([0.68257619, 0.6693502 , 0.68602645, 0.71576525, 0.67146145])}

In [34]:
log_baseline_cv['test_score'].mean()


0.6850359087633529

In [35]:
log_pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('ordinal_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('ordinal_encoded',
                                                                   OrdinalEncoder(categories=[['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'Europa',
                                                                                               'Earth',
                                                                                               'Mars'],
                                                                                              ['missing',
                                                                                               'TRAPPIST-1e',
                                                                                               'PSO '
                                                                                               'J318.5...
                                                                   MinMaxScaler())]),
                                                  ['Age', 'FoodCourt',
                                                   'RoomService',
                                                   'ShoppingMall', 'Spa',
                                                   'VRDeck']),
                                                 ('categorical_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='None',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(drop='if_binary',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Cabin', 'Name',
                                                   'PassengerId'])])),
                ('modeling', LogisticRegression(random_state=42))])

In [46]:
log_random_search = RandomizedSearchCV(
    log_pipe,
    param_distributions={
        'modeling__solver': ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga'],
        'modeling__C': randint(1, 16)
    },
    cv=5,
    n_iter=10,
    scoring='accuracy',
    n_jobs=-1
)

log_random_search.fit(X_train, y_train)

/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('ordinal_preprocessor',
                                                                               Pipeline(steps=[('nan_imputer',
                                                                                                SimpleImputer(fill_value='missing',
                                                                                                              strategy='constant')),
                                                                                               ('ordinal_encoded',
                                                                                                OrdinalEncoder(categories=[['missing',
                                                                                                                            'False',
                                                                                                                            'True'],
                                                                                                                           ['missing',
                                                                                                                            'False',
                                                                                                                            'True'],
                                                                                                                           ['missing',
                                                                                                                            'Europa',
                                                                                                                            'Earth',
                                                                                                                            'Mars'],
                                                                                                                           ['m...
                                                                                                OneHotEncoder(drop='if_binary',
                                                                                                              handle_unknown='ignore'))]),
                                                                               ['Cabin',
                                                                                'Name',
                                                                                'PassengerId'])])),
                                             ('modeling',
                                              LogisticRegression(random_state=42))]),
                   n_jobs=-1,
                   param_distributions={'modeling__C': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ff80d867c20>,
                                        'modeling__solver': ['lbfgs',
                                                             'liblinear',
                                                             'newton-cg', 'sag',
                                                             'saga']},
                   scoring='accuracy')

In [47]:
log_random_search.best_score_

0.7039024186883062

In [48]:
log_random_search.best_params_

{'modeling__C': 14, 'modeling__solver': 'lbfgs'}

In [49]:
log_tuned_pipe = log_random_search.best_estimator_

In [50]:
log_tuned_pipe

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('ordinal_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('ordinal_encoded',
                                                                   OrdinalEncoder(categories=[['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'Europa',
                                                                                               'Earth',
                                                                                               'Mars'],
                                                                                              ['missing',
                                                                                               'TRAPPIST-1e',
                                                                                               'PSO '
                                                                                               'J318.5...
                                                                   MinMaxScaler())]),
                                                  ['Age', 'FoodCourt',
                                                   'RoomService',
                                                   'ShoppingMall', 'Spa',
                                                   'VRDeck']),
                                                 ('categorical_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='None',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(drop='if_binary',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Cabin', 'Name',
                                                   'PassengerId'])])),
                ('modeling', LogisticRegression(C=14, random_state=42))])

## RandomForest

In [59]:
rf_pipe = Pipeline([
    ('preprocessing', preproc_pipe),
    ('modeling', RandomForestClassifier(n_jobs=-1, random_state=42))
])

rf_pipe

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('ordinal_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('ordinal_encoded',
                                                                   OrdinalEncoder(categories=[['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'Europa',
                                                                                               'Earth',
                                                                                               'Mars'],
                                                                                              ['missing',
                                                                                               'TRAPPIST-1e',
                                                                                               'PSO '
                                                                                               'J318.5...
                                                                   MinMaxScaler())]),
                                                  ['Age', 'FoodCourt',
                                                   'RoomService',
                                                   'ShoppingMall', 'Spa',
                                                   'VRDeck']),
                                                 ('categorical_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='None',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(drop='if_binary',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Cabin', 'Name',
                                                   'PassengerId'])])),
                ('modeling',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])

## RF Baseline score

In [62]:
rf_baseline_cv = cross_validate(rf_pipe,
                                X_train, y_train,
                                cv=5,
                                scoring='accuracy',
                                n_jobs=-1)

/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as

In [61]:
rf_baseline_cv['test_score'].mean()

0.783160963769636

In [63]:
rf_pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('ordinal_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('ordinal_encoded',
                                                                   OrdinalEncoder(categories=[['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'Europa',
                                                                                               'Earth',
                                                                                               'Mars'],
                                                                                              ['missing',
                                                                                               'TRAPPIST-1e',
                                                                                               'PSO '
                                                                                               'J318.5...
                                                                   MinMaxScaler())]),
                                                  ['Age', 'FoodCourt',
                                                   'RoomService',
                                                   'ShoppingMall', 'Spa',
                                                   'VRDeck']),
                                                 ('categorical_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='None',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(drop='if_binary',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Cabin', 'Name',
                                                   'PassengerId'])])),
                ('modeling',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])

## True RandomForestCl

In [65]:
rf_random_search = RandomizedSearchCV(
    rf_pipe,
    param_distributions={
        'modeling__max_depth': randint(2,50),
        'modeling__min_samples_split': randint(2, 20),
        'modeling__min_samples_leaf': randint(0, 10),
        'modeling__n_estimators': randint(1, 200),
        'modeling__criterion': ['gini', 'entropy', 'log_loss'],
        'modeling__max_features': ['sqrt', 'log2', None],
        'modeling__bootstrap': [True, False],
    },
    cv=5,
    n_iter=10,
    scoring='accuracy',
    n_jobs=-1
)

rf_random_search.fit(X_train, y_train)

/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('ordinal_preprocessor',
                                                                               Pipeline(steps=[('nan_imputer',
                                                                                                SimpleImputer(fill_value='missing',
                                                                                                              strategy='constant')),
                                                                                               ('ordinal_encoded',
                                                                                                OrdinalEncoder(categories=[['missing',
                                                                                                                            'False',
                                                                                                                            'True'],
                                                                                                                           ['missing',
                                                                                                                            'False',
                                                                                                                            'True'],
                                                                                                                           ['missing',
                                                                                                                            'Europa',
                                                                                                                            'Earth',
                                                                                                                            'Mars'],
                                                                                                                           ['m...
                                        'modeling__max_features': ['sqrt',
                                                                   'log2',
                                                                   None],
                                        'modeling__min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ff80d877590>,
                                        'modeling__min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ff80d877e60>,
                                        'modeling__n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ff80d760e30>},
                   scoring='accuracy')

In [66]:
rf_random_search.best_score_

0.750833746362968

In [67]:
rf_random_search.best_params_

{'modeling__bootstrap': False,
 'modeling__criterion': 'log_loss',
 'modeling__max_depth': 39,
 'modeling__max_features': None,
 'modeling__min_samples_leaf': 9,
 'modeling__min_samples_split': 3,
 'modeling__n_estimators': 77}

In [68]:
rf_tuned_pipe = rf_random_search.best_estimator_

In [69]:
rf_tuned_pipe

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('ordinal_preprocessor',
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('ordinal_encoded',
                                                                   OrdinalEncoder(categories=[['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'False',
                                                                                               'True'],
                                                                                              ['missing',
                                                                                               'Europa',
                                                                                               'Earth',
                                                                                               'Mars'],
                                                                                              ['missing',
                                                                                               'TRAPPIST-1e',
                                                                                               'PSO '
                                                                                               'J318.5...
                                                  Pipeline(steps=[('nan_imputer',
                                                                   SimpleImputer(fill_value='None',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(drop='if_binary',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Cabin', 'Name',
                                                   'PassengerId'])])),
                ('modeling',
                 RandomForestClassifier(bootstrap=False, criterion='log_loss',
                                        max_depth=39, max_features=None,
                                        min_samples_leaf=9, min_samples_split=3,
                                        n_estimators=77, n_jobs=-1,
                                        random_state=42))])

## Export

In [73]:
rf_pipe.fit(X_train, y_train)

X_test = pd.read_csv('data/titanic_test.csv')

predictions = rf_pipe.predict(X_test)

results = pd.concat([X_test["PassengerId"], pd.Series(predictions, name="Transported")], axis=1)

results.to_csv("data/submission_final.csv", header=True, index=False)

/home/guillaume/.pyenv/versions/lewagon/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
